# LLM Agents Test Notebook

This notebook exercises the fresh project's LLM-enabled agents:

- `TechnicalAnalysisAgent`
- `NewsResearchAgent`
- `RiskReviewAgent`
- `DecisionCoordinatorAgent`
- End-to-end `TradingWorkflow`

Fill in `API_KEY` below or export `DEEPSEEK_API_KEY` before running the live LLM cells.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "fresh_simple_trading_project":
    matching_root = next((path for path in [PROJECT_ROOT, *PROJECT_ROOT.parents] if path.name == "fresh_simple_trading_project"), None)
    if matching_root is None:
        raise RuntimeError("Run this notebook from inside the fresh_simple_trading_project directory tree.")
    PROJECT_ROOT = matching_root

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

API_KEY = ""  # Paste your DeepSeek/OpenAI-compatible API key here if you do not want to use DEEPSEEK_API_KEY.
BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-reasoner"
TIMEOUT_SECONDS = 45.0
TEST_SYMBOL = "AAPL"

EFFECTIVE_API_KEY = API_KEY.strip() or os.getenv("DEEPSEEK_API_KEY", "").strip()
print({
    "project_root": str(PROJECT_ROOT),
    "src_path": str(SRC_PATH),
    "has_api_key": bool(EFFECTIVE_API_KEY),
    "base_url": BASE_URL,
    "model": MODEL,
    "symbol": TEST_SYMBOL,
})

In [ ]:
from dataclasses import asdict

import numpy as np
import pandas as pd

from fresh_simple_trading_project.agents import DecisionCoordinatorAgent
from fresh_simple_trading_project.alpaca_integration import AlpacaService
from fresh_simple_trading_project.config import LLMConfig, Settings
from fresh_simple_trading_project.data_collection import DataCollectionModule, StaticAccountClient, StaticMarketDataClient
from fresh_simple_trading_project.decision_engine import DecisionEngine
from fresh_simple_trading_project.eda import EDAModule
from fresh_simple_trading_project.execution import ExecutionModule, InMemoryBrokerClient
from fresh_simple_trading_project.features import FeatureEngineeringModule
from fresh_simple_trading_project.information_retrieval import (
    AlpacaNewsSearchClient,
    CombinedNewsSearchClient,
    InformationRetrievalModule,
    StaticNewsSearchClient,
    WebSearchNewsClient,
)
from fresh_simple_trading_project.llm import DeepSeekLLMClient
from fresh_simple_trading_project.market_analysis import MarketAnalysisModule
from fresh_simple_trading_project.models import AccountState, NewsArticle
from fresh_simple_trading_project.risk_analysis import RiskAnalysisModule
from fresh_simple_trading_project.storage import InMemoryResultStore
from fresh_simple_trading_project.workflow import TradingWorkflow


In [ ]:
def make_llm_client() -> DeepSeekLLMClient:
    if not EFFECTIVE_API_KEY:
        raise ValueError("Set API_KEY in the notebook or export DEEPSEEK_API_KEY before running live LLM tests.")
    return DeepSeekLLMClient(
        LLMConfig(
            api_key=EFFECTIVE_API_KEY,
            base_url=BASE_URL,
            model=MODEL,
            timeout_seconds=TIMEOUT_SECONDS,
        )
    )


def make_demo_bars(periods: int = 720) -> pd.DataFrame:
    index = pd.date_range("2025-01-01", periods=periods, freq="5min", tz="UTC")
    base = np.linspace(100.0, 160.0, num=len(index))
    return pd.DataFrame(
        {
            "open": base - 0.5,
            "high": base + 1.0,
            "low": base - 1.0,
            "close": base,
            "volume": np.full(len(index), 50_000),
        },
        index=index,
    )


def make_demo_articles(symbol: str) -> list[NewsArticle]:
    return [
        NewsArticle(
            headline=f"{symbol} earnings beat expectations with strong growth",
            summary="Analysts point to margin expansion and resilient demand.",
            source="Notebook News",
            published_at="2025-01-01T12:00:00Z",
        ),
        NewsArticle(
            headline=f"{symbol} receives analyst upgrade after guidance raise",
            summary="Follow-through buying improves after upbeat management commentary.",
            source="Notebook News",
            published_at="2025-01-01T13:00:00Z",
        ),
    ]


class NoOpRawStore:
    def save_bars(self, symbol: str, timeframe: str, bars: pd.DataFrame):
        return None

    def save_news(self, symbol: str, articles: list[NewsArticle]):
        return None


settings = Settings.from_env(project_root=PROJECT_ROOT)
bars = make_demo_bars()
articles = make_demo_articles(TEST_SYMBOL)
account = AccountState(cash=settings.trading.starting_cash, position_qty=0, market_open=True)
llm_client = make_llm_client()

print({
    "bars": len(bars),
    "articles": len(articles),
    "market_data_provider": settings.market_data.provider,
    "alpaca_news_enabled": settings.alpaca.enabled,
    "live_web_search_enabled": True,
    "news_max_age_days": settings.news.max_age_days,
    "llm_enabled": llm_client.enabled,
})


## 1. Technical Analysis Agent Test

This runs `MarketAnalysisModule`, which internally triggers `TechnicalAnalysisAgent` with a plain-text market snapshot prompt.

In [ ]:
feature_engineering = FeatureEngineeringModule(settings.trading)
feature_frame = feature_engineering.build(bars)
market_analysis_module = MarketAnalysisModule(settings.trading, llm_client=llm_client)
analysis = market_analysis_module.analyze(TEST_SYMBOL, feature_frame)

print({
    "trend": analysis.trend,
    "latest_price": analysis.latest_price,
    "entry_setup": analysis.entry_setup,
    "exit_setup": analysis.exit_setup,
    "technical_agent_summary": analysis.llm_summary,
})
analysis.notes[-5:]

## 2. News Research Agent Test

This first runs `InformationRetrievalModule` with static notebook articles. The next cell verifies the live hybrid path by calling both Alpaca news and the live web-search feed before running `NewsResearchAgent` on the merged results.


In [ ]:
information_retrieval_module = InformationRetrievalModule(
    StaticNewsSearchClient(articles),
    llm_client=llm_client,
)
retrieval = information_retrieval_module.retrieve(TEST_SYMBOL, limit=3)

print({
    "sentiment_score": retrieval.sentiment_score,
    "catalysts": retrieval.catalysts,
    "risk_flags": retrieval.risk_flags,
    "news_agent_summary": retrieval.summary_note,
})
retrieval.headline_summary

### 2A. Live Hybrid News Source Verification

This cell fetches live news from both sources separately, asserts that both return at least one article, then runs `InformationRetrievalModule` with the combined client stack that the fresh project uses in the workflow.


In [ ]:
if not settings.alpaca.enabled:
    raise ValueError(
        "Set ALPACA_PAPER_API_KEY and ALPACA_PAPER_SECRET_KEY in fresh_simple_trading_project/.env "
        "to verify Alpaca + live web-search news together."
    )

alpaca_service = AlpacaService(settings.alpaca)
alpaca_news_client = AlpacaNewsSearchClient(alpaca_service, max_age_days=settings.news.max_age_days)
web_search_news_client = WebSearchNewsClient(max_age_days=settings.news.max_age_days)
combined_live_news_client = CombinedNewsSearchClient([alpaca_news_client, web_search_news_client])

live_news_queries = {
    "symbol": f"{TEST_SYMBOL} stock market news",
    "macro": "American economy inflation interest rates jobs GDP Federal Reserve",
    "market": "overall U.S. stock market trend S&P 500 Nasdaq Dow market breadth",
}

live_news_batches = {
    "alpaca": {
        scope: alpaca_news_client.search_news(query, limit=5)
        for scope, query in live_news_queries.items()
    },
    "web_search": {
        scope: web_search_news_client.search_news(query, limit=5)
        for scope, query in live_news_queries.items()
    },
    "combined": {
        scope: combined_live_news_client.search_news(query, limit=8)
        for scope, query in live_news_queries.items()
    },
}

alpaca_total = sum(len(items) for items in live_news_batches["alpaca"].values())
web_search_total = sum(len(items) for items in live_news_batches["web_search"].values())
if alpaca_total == 0:
    raise AssertionError("Alpaca live news returned 0 articles across the test queries.")
if web_search_total == 0:
    raise AssertionError("Live web-search news returned 0 articles across the test queries.")

live_information_retrieval_module = InformationRetrievalModule(
    combined_live_news_client,
    llm_client=llm_client,
    max_article_age_days=settings.news.max_age_days,
)
live_retrieval = live_information_retrieval_module.retrieve(TEST_SYMBOL, limit=8)

print({
    "alpaca_total_articles": alpaca_total,
    "web_search_total_articles": web_search_total,
    "combined_articles_used": len(live_retrieval.articles),
    "news_agent_summary": live_retrieval.summary_note,
})

scope_counts = pd.DataFrame(
    [
        {
            "source_client": source_name,
            "scope": scope,
            "article_count": len(scope_articles),
        }
        for source_name, source_batches in live_news_batches.items()
        for scope, scope_articles in source_batches.items()
    ]
)
print(scope_counts.to_string(index=False))

live_news_rows = [
    {
        "source_client": source_name,
        "scope": scope,
        "headline": article.headline,
        "publisher": article.source,
        "published_at": article.published_at,
        "url": article.url,
    }
    for source_name, source_batches in live_news_batches.items()
    for scope, scope_articles in source_batches.items()
    for article in scope_articles
]

pd.DataFrame(live_news_rows)


## 3. Risk Review Agent Test

This runs `RiskAnalysisModule`, which internally triggers `RiskReviewAgent`.

In [ ]:
eda_module = EDAModule()
eda = eda_module.summarize(TEST_SYMBOL, bars.tail(settings.trading.eda_window_hours * 12))
risk_analysis_module = RiskAnalysisModule(settings.trading, llm_client=llm_client)
risk = risk_analysis_module.assess(TEST_SYMBOL, account, analysis, eda, retrieval)

print({
    "risk_score": risk.risk_score,
    "can_enter": risk.can_enter,
    "recommended_qty": risk.recommended_qty,
    "warnings": risk.warnings,
    "risk_agent_summary": risk.summary_note,
})

## 4. Decision Coordinator Agent Test

This calls `DecisionCoordinatorAgent` directly. It reviews plain-text market, news, and risk context before returning the final review.

In [ ]:
decision_agent = DecisionCoordinatorAgent(llm_client=llm_client)
decision_review = decision_agent.review(
    rule_based_action="BUY",
    rule_based_quantity=risk.recommended_qty,
    account=account,
    analysis=analysis,
    retrieval=retrieval,
    risk=risk,
)

asdict(decision_review)

## 5. End-to-End Workflow Smoke Test

This builds a standalone workflow with static market/news inputs, in-memory result storage, and no-op raw storage to avoid writing files during notebook experimentation.

In [ ]:
workflow = TradingWorkflow(
    settings=settings,
    data_collection=DataCollectionModule(
        market_data_client=StaticMarketDataClient(bars),
        account_client=StaticAccountClient(cash=settings.trading.starting_cash),
    ),
    eda_module=EDAModule(),
    feature_engineering=FeatureEngineeringModule(settings.trading),
    market_analysis=MarketAnalysisModule(settings.trading, llm_client=llm_client),
    information_retrieval=InformationRetrievalModule(
        StaticNewsSearchClient(articles),
        llm_client=llm_client,
    ),
    risk_analysis=RiskAnalysisModule(settings.trading, llm_client=llm_client),
    decision_engine=DecisionEngine(llm_client=llm_client),
    execution_module=ExecutionModule(InMemoryBrokerClient()),
    raw_store=NoOpRawStore(),
    result_store=InMemoryResultStore(),
)

workflow_result = workflow.run_once(symbol=TEST_SYMBOL, execute_orders=False)

workflow_summary = {
    "symbol": workflow_result.symbol,
    "technical_agent": workflow_result.analysis.llm_summary,
    "news_agent": workflow_result.retrieval.summary_note,
    "risk_agent": workflow_result.risk.summary_note,
    "action": workflow_result.decision.action.value,
    "quantity": workflow_result.decision.quantity,
    "confidence": workflow_result.decision.confidence,
    "execution_status": workflow_result.execution.status,
    "decision_rationale": workflow_result.decision.rationale,
}
workflow_summary

In [ ]:
def show_reasoning(label: str, text: str | None) -> None:
    print(f"\n=== {label} ===")
    print(text.strip() if text else "<no output returned>")


show_reasoning("Technical Agent", analysis.llm_summary)
show_reasoning("News Agent", retrieval.summary_note)
show_reasoning("Risk Agent", risk.summary_note)

print("\n=== Decision Agent Review ===")
print({
    "action": decision_review.action,
    "quantity": decision_review.quantity,
    "note": decision_review.note,
    "override": decision_review.override,
})

print("\n=== Workflow Decision Rationale ===")
for idx, line in enumerate(workflow_result.decision.rationale, 1):
    print(f"{idx}. {line}")

print("\n=== Workflow Summary ===")
print({
    "technical_agent": workflow_result.analysis.llm_summary,
    "news_agent": workflow_result.retrieval.summary_note,
    "risk_agent": workflow_result.risk.summary_note,
    "action": workflow_result.decision.action.value,
    "quantity": workflow_result.decision.quantity,
    "confidence": workflow_result.decision.confidence,
    "execution_status": workflow_result.execution.status,
})


In [ ]:
# result = workflow.run_once(symbol="AAPL", execute_orders=False)

In [ ]:
import pandas as pd

pd.DataFrame([article.__dict__ for article in result.retrieval.articles])


In [ ]:
from fresh_simple_trading_project.workflow import format_reasoning_lines

print("\n".join(format_reasoning_lines(result, prefix="")))


In [ ]:
print(
    f"Data Window: {result.five_minute_bars.index.min().isoformat()} -> "
    f"{result.five_minute_bars.index.max().isoformat()}"
)


In [ ]:
from pathlib import Path
import pandas as pd

from fresh_simple_trading_project.workflow import build_workflow

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
workflow = build_workflow(project_root=project_root)

symbol = "AAPL"

collected = workflow.data_collection.collect(
    symbol=symbol,
    lookback_hours=workflow.settings.trading.lookback_hours,
)

print(f"Symbol: {symbol}")
print(f"5-minute bars: {len(collected.five_minute_bars)}")
print(f"Hourly bars: {len(collected.hourly_bars)}")
print(
    "5-minute window:",
    collected.five_minute_bars.index.min().isoformat(),
    "->",
    collected.five_minute_bars.index.max().isoformat(),
)
print(
    "Hourly window:",
    collected.hourly_bars.index.min().isoformat(),
    "->",
    collected.hourly_bars.index.max().isoformat(),
)

collected.five_minute_bars.tail()


## Optional Next Steps

- Swap `TEST_SYMBOL` to another symbol.
- Set `MARKET_DATA_PROVIDER=alpaca` if you also want live Alpaca bars and paper-account state in the notebook workflow test.
- Compare the static news-agent output with the live hybrid news-agent output above.
- Run the notebook twice with different prompts/models and compare the decision rationale.
